# Phase 2A — Sparse vs Dense: memory and custom dot products

**Assignment:** 1K/5K/10K/20K corpus sizes; TF-IDF CSR and real dense embeddings; implement sparse–sparse, sparse–dense and dense–dense dot products ourselves; measure time/memory; justify representation.

**Important experimental design:** TF-IDF and semantic embeddings occupy different feature spaces. We measure their memory and query-search performance separately. For the *three dot-product implementation* comparisons, we create a **matched-dimensional** experimental matrix: dense semantic embeddings and a CSR copy of those same embeddings (exact, not thresholded). This makes each multiplication mathematically meaningful and permits correctness checks. **CSR indices must be sorted for the custom two-pointer sparse–sparse algorithm; the notebook sorts them explicitly before testing.** The CSR copy may not save memory because dense embeddings contain few zeros — this itself is an important result.

The notebook runs a 1K smoke test first. Increase `SCALES` to the required four sizes after the first successful run. No timings or figures are fabricated. All compute benchmarks are CPU-only.

In [ ]:
%pip -q install "sentence-transformers>=3,<6" scikit-learn scipy numpy pandas matplotlib psutil

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, time, csv, statistics, platform, sys, gc
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import torch

PROJECT=Path('/content/drive/MyDrive/HP-SS-Engine')
DATA=PROJECT/'data/raw/qwen_corpus_20k_progress.jsonl'
OUT=PROJECT/'phase2_matrix_ops'
BENCH=OUT/'benchmarks'
FIG=OUT/'figures'
EMB=OUT/'2A_sparse_dense'
for d in (OUT,BENCH,FIG,EMB): d.mkdir(parents=True,exist_ok=True)
assert DATA.exists(),f'Missing corpus at {DATA}. Check Drive folder/path.'
records=[json.loads(line) for line in DATA.open(encoding='utf-8') if line.strip()]
records.sort(key=lambda x:x['id'])
assert len(records)==20000 and [x['id'] for x in records]==list(range(20000))
texts=[r['response'] for r in records]
queries=[records[i]['prompt'] for i in range(0,20000,200)][:100]
print('Documents:',len(texts),'CPU:',platform.processor(),'CPU count:',os.cpu_count())
print('NumPy',np.__version__,'SciPy',scipy.__version__,'Torch',torch.__version__)
print('SentenceTransformer',SentenceTransformer.__module__)
print('Embedding inference may use GPU if available; search timings are CPU.')

## Step 1 — Build two genuine representations

- **TF-IDF CSR:** sparse word-frequency importance matrix.
- **Semantic embeddings:** a pretrained model maps each text to a 384-dimensional dense vector.

Embeddings are saved in Drive to avoid recomputing them every session. Corpus order is always ID-sorted. The cached embedding array is reused only if shape and metadata match.

In [ ]:
MODEL_NAME='sentence-transformers/all-MiniLM-L6-v2'
EMB_PATH=EMB/'qwen_20k_minilm_384_float32.npy'
META_PATH=EMB/'qwen_20k_minilm_metadata.json'
model=SentenceTransformer(MODEL_NAME,device='cuda' if torch.cuda.is_available() else 'cpu')
dim=model.get_sentence_embedding_dimension()
print('Embedding model:',MODEL_NAME,'dimensions:',dim,'device:',model.device)
expected={'model':MODEL_NAME,'rows':len(texts),'dimension':dim}
cached=False
if EMB_PATH.exists() and META_PATH.exists():
    meta=json.loads(META_PATH.read_text(encoding='utf-8'))
    cached=all(meta.get(k)==v for k,v in expected.items())
if cached:
    E=np.load(EMB_PATH,mmap_mode='r')
    assert E.shape==(len(texts),dim)
    print('Loaded cached embeddings:',E.shape)
else:
    print('Generating real embeddings. This may take several minutes.')
    # Stream to disk: if interrupted, the incomplete file will not be considered a valid cache.
    tmp=EMB/'qwen_20k_minilm_partial.npy'
    arr=np.lib.format.open_memmap(tmp,mode='w+',dtype=np.float32,shape=(len(texts),dim))
    batch_size=128
    for start in range(0,len(texts),batch_size):
        end=min(start+batch_size,len(texts))
        arr[start:end]=model.encode(texts[start:end],batch_size=64,convert_to_numpy=True,
            normalize_embeddings=True,show_progress_bar=False).astype(np.float32)
        if end%2048==0 or end==len(texts): print('Embedded',end,'/',len(texts))
    arr.flush();del arr
    os.replace(tmp,EMB_PATH)
    META_PATH.write_text(json.dumps(expected,indent=2),encoding='utf-8')
    E=np.load(EMB_PATH,mmap_mode='r')
Q_E=model.encode(queries,batch_size=64,convert_to_numpy=True,
    normalize_embeddings=True,show_progress_bar=False).astype(np.float32)
assert E.shape==(20000,dim) and Q_E.shape==(100,dim)
print('Dense embeddings ready:',E.shape)

In [ ]:
vectorizer=TfidfVectorizer(dtype=np.float32,norm='l2',min_df=1)
X_TFIDF=vectorizer.fit_transform(texts).tocsr()
Q_TFIDF=vectorizer.transform(queries).tocsr()
# Two-pointer sparse dot requires sorted column indices within every CSR row.
# sklearn TF-IDF can produce CSR rows with unsorted indices.
X_TFIDF.sort_indices()
Q_TFIDF.sort_indices()
assert X_TFIDF.has_sorted_indices and Q_TFIDF.has_sorted_indices
assert X_TFIDF.shape[0]==20000 and Q_TFIDF.shape[0]==100
print('Sparse TF-IDF shape:',X_TFIDF.shape,'nnz:',X_TFIDF.nnz)
print('Dense embedding shape:',E.shape)
def csr_bytes(x):
    return x.data.nbytes+x.indices.nbytes+x.indptr.nbytes
print('20K TF-IDF CSR MiB:',round(csr_bytes(X_TFIDF)/2**20,2))
print('20K embedding MiB:',round(E.nbytes/2**20,2))

## Step 2 — Implement dot products ourselves

The following functions **do not use sklearn similarity helpers or `@`**. The sparse–sparse function uses a two-pointer scan of sorted nonzero indices; sparse–dense accesses only the sparse row's nonzero entries; dense–dense explicitly multiplies and sums matching elements with NumPy (no built-in dot product).

For a fair three-way *arithmetic* check, sparse and dense inputs must describe the **same 384-dimensional embedding coordinates**. A CSR representation of dense embeddings is expected to be inefficient in memory, since embeddings are not sparse.

In [ ]:
def dot_sparse_sparse(a_idx,a_val,b_idx,b_val):
    """Own two-pointer dot product of sorted sparse index/value pairs."""
    i=j=0
    total=0.0
    while i<len(a_idx) and j<len(b_idx):
        if a_idx[i]==b_idx[j]:
            total+=float(a_val[i])*float(b_val[j])
            i+=1;j+=1
        elif a_idx[i]<b_idx[j]:
            i+=1
        else:
            j+=1
    return total

def dot_sparse_dense(a_idx,a_val,b_dense):
    """Own dot product: sparse vector vs dense vector."""
    total=0.0
    for t in range(len(a_idx)):
        total+=float(a_val[t])*float(b_dense[a_idx[t]])
    return total

def dot_dense_dense(a_dense,b_dense):
    """Own element-wise multiply-and-sum, no @ or np.dot."""
    return float(np.sum(a_dense*b_dense,dtype=np.float64))

def csr_row_parts(mat,i):
    assert mat.has_sorted_indices, 'CSR column indices must be sorted; call matrix.sort_indices() first.'
    start,end=mat.indptr[i],mat.indptr[i+1]
    return mat.indices[start:end],mat.data[start:end]

# A separate real TF-IDF sparse-sparse correctness check
for i in [0,1,2,9]:
    ai,av=csr_row_parts(X_TFIDF,i)
    bi,bv=csr_row_parts(Q_TFIDF,0)
    actual=dot_sparse_sparse(ai,av,bi,bv)
    expected=float(X_TFIDF.getrow(i).multiply(Q_TFIDF.getrow(0)).sum())
    assert abs(actual-expected)<1e-5,(i,actual,expected)

# Matched-space correctness checks: same semantic embeddings as dense or CSR
D_TEST=np.asarray(E[:10],dtype=np.float32)
C_TEST=sparse.csr_matrix(D_TEST)
q=np.asarray(Q_E[0],dtype=np.float32)
qcsr=sparse.csr_matrix(q.reshape(1,-1))
for i in range(10):
    ai,av=csr_row_parts(C_TEST,i)
    bi,bv=csr_row_parts(qcsr,0)
    ref=float(np.dot(D_TEST[i],q))
    for got in (dot_sparse_sparse(ai,av,bi,bv),
                dot_sparse_dense(ai,av,q),
                dot_dense_dense(D_TEST[i],q)):
        assert np.isclose(got,ref,atol=1e-5,rtol=1e-5),(i,got,ref)
print('Correctness passed: all 3 manual dot products match reference within 1e-5')

## Step 3 — Pilot benchmark, then required scales

Start with `SCALES=[1000]`. Once correct, set `SCALES=[1000,5000,10000,20000]` and rerun this cell and the plotting cell.

The full benchmark has **5 queries × 5 repeats per scale × 3 dot-product methods**, and manually iterates over documents. For 20K documents this can take time. A pilot at 1K makes sure no expensive long run is wasted.

Each timed query computes scores against all `n` documents, then top-10. Setup/conversion is outside timing. `dense_dense` uses vectorized element-wise NumPy multiplication per document, not a matrix `@` call. An optimized matrix multiplication will be benchmarked separately in Phase 2C.

In [ ]:
def topk_from_scores(scores,k=10):
    scores=np.asarray(scores,dtype=np.float64)
    ids=np.lexsort((np.arange(len(scores)),-scores))[:k]
    return [(int(i),float(scores[i])) for i in ids]

def search_sparse_sparse(mat,qmat,k=10):
    qi,qv=csr_row_parts(qmat,0)
    scores=[]
    for i in range(mat.shape[0]):
        ai,av=csr_row_parts(mat,i)
        scores.append(dot_sparse_sparse(ai,av,qi,qv))
    return topk_from_scores(scores,k)

def search_sparse_dense(mat,q,k=10):
    scores=[]
    for i in range(mat.shape[0]):
        ai,av=csr_row_parts(mat,i)
        scores.append(dot_sparse_dense(ai,av,q))
    return topk_from_scores(scores,k)

def search_dense_dense(mat,q,k=10):
    scores=[]
    for i in range(len(mat)):
        scores.append(dot_dense_dense(mat[i],q))
    return topk_from_scores(scores,k)

# Use actual TF-IDF for sparse-sparse; matched semantic embedding CSR for
# sparse-dense and dense-dense. Do NOT treat latency differences as pure
# storage-format speedups: the first method uses a different representation.
SCALES=[1000] # After pilot: [1000,5000,10000,20000]
QUERY_IDS=list(range(5))
REPEATS=5
raw=[]
memory=[]
for n in SCALES:
    tfidf=X_TFIDF[:n]
    dense=np.asarray(E[:n],dtype=np.float32)
    dense_csr=sparse.csr_matrix(dense)
    memory.append({'n_docs':n,
        'tfidf_csr_mib':csr_bytes(tfidf)/2**20,
        'embedding_dense_mib':dense.nbytes/2**20,
        'embedding_csr_mib':csr_bytes(dense_csr)/2**20,
        'tfidf_features':tfidf.shape[1],'embedding_features':dense.shape[1],
        'tfidf_nnz':tfidf.nnz,'embedding_csr_nnz':dense_csr.nnz})
    # 3 separately named workloads. The dense and embedding-CSR share exact values.
    for qi in QUERY_IDS:
        query_tfidf=Q_TFIDF[qi]
        query_dense=np.asarray(Q_E[qi],dtype=np.float32)
        query_dense_csr=sparse.csr_matrix(query_dense.reshape(1,-1))
        cases=[
            ('sparse_sparse_tfidf',lambda:search_sparse_sparse(tfidf,query_tfidf)),
            ('sparse_dense_embedding',lambda:search_sparse_dense(dense_csr,query_dense)),
            ('dense_dense_embedding',lambda:search_dense_dense(dense,query_dense))]
        for method,fn in cases:
            if qi==0: fn() # warmup, not timed
            for rep in range(REPEATS):
                t0=time.perf_counter_ns()
                got=fn()
                ms=(time.perf_counter_ns()-t0)/1e6
                raw.append({'method':method,'n_docs':n,'query_id':qi,'repeat':rep,
                    'latency_ms':ms,'top_k':10})
    print('Finished',n,'docs')
    del dense_csr,dense;gc.collect()

raw_df=pd.DataFrame(raw)
memory_df=pd.DataFrame(memory)
raw_df.to_csv(BENCH/'phase2a_dot_products_raw.csv',index=False)
memory_df.to_csv(BENCH/'phase2a_memory.csv',index=False)
print(raw_df.groupby(['n_docs','method'])['latency_ms'].agg(['count','median','mean','std']).round(3))
print(memory_df.to_string(index=False))
print('Saved:',BENCH)

## Step 4 — Figures and decision evidence

The first figure compares **memory footprint** of real TF-IDF CSR versus real dense embeddings; the second shows timings of the three manually implemented dot products. The mixed workload caveat appears in its caption. Figures are generated from measured CSVs only.

In [ ]:
assert len(memory_df)>0 and len(raw_df)>0
fig,ax=plt.subplots(figsize=(8,5))
for col,label in [('tfidf_csr_mib','TF-IDF CSR'),('embedding_dense_mib','Semantic embeddings dense'),('embedding_csr_mib','Semantic embeddings stored as CSR')]:
    ax.plot(memory_df['n_docs'],memory_df[col],marker='o',label=label)
ax.set(title='Phase 2A — Memory by corpus size',xlabel='Documents',ylabel='Storage (MiB)')
ax.legend();ax.grid(alpha=.25);fig.tight_layout()
p=FIG/'phase2a_memory.png';fig.savefig(p,dpi=180);plt.show();plt.close(fig)

summ=raw_df.groupby(['n_docs','method'],as_index=False)['latency_ms'].median()
fig,ax=plt.subplots(figsize=(9,5))
for name,part in summ.groupby('method'):
    ax.plot(part['n_docs'],part['latency_ms'],marker='o',label=name)
ax.set(title='Phase 2A — Manual dot products: median search time',
       xlabel='Documents',ylabel='Latency per query (ms)')
ax.legend(fontsize=8);ax.grid(alpha=.25);fig.tight_layout()
p=FIG/'phase2a_manual_dot_latency.png';fig.savefig(p,dpi=180);plt.show();plt.close(fig)
print('Figures saved to',FIG)
print('Interpretation warning: TF-IDF and semantic embeddings differ in feature space and sparsity; compare each workload on its own merits, not as a pure representation-only speedup.')

## Before moving to Phase 2B

1. Confirm `Correctness passed`.
2. Rerun with all four required corpus sizes and verify the saved CSV has 4×5×5×4 = **400 timing rows**.
3. Share the median timing table and memory table; we will record hardware, discuss the right representation and add report-ready figure captions.
4. Do **not** regenerate embeddings unnecessarily: the model output is cached in Drive.
5. The three dot-product implementations are independent and visible above; the extra embedding sparse–sparse case lets us compare all three storage combinations on the *same* semantic vectors.